In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import confusion_matrix, accuracy_score

In [31]:
df = pd.read_csv("accidentsFull.csv")

df.head()

,HOUR_I_R,ALCHL_I,ALIGN_I,STRATUM_R,WRK_ZONE,WKDY_I_R,INT_HWY,LGTCON_I_R,MANCOL_I_R,PED_ACC_R,...,SUR_COND,TRAF_CON_R,TRAF_WAY,VEH_INVL,WEATHER_R,INJURY_CRASH,NO_INJ_I,PRPTYDMG_CRASH,FATALITIES,MAX_SEV_IR
0,0,2,2,1,0,1,0,3,0,0,...,4,0,3,1,1,1,1,0,0,1
1,1,2,1,0,0,1,1,3,2,0,...,4,0,3,2,2,0,0,1,0,0
2,1,2,1,0,0,1,0,3,2,0,...,4,1,2,2,2,0,0,1,0,0
3,1,2,1,1,0,0,0,3,2,0,...,4,1,2,2,1,0,0,1,0,0
4,1,1,1,0,0,1,0,3,2,0,...,4,0,2,3,1,0,0,1,0,0


In [32]:
df['INJURY'] = np.where(df['MAX_SEV_IR'].isin([1,2]), "Yes", "No")

a)

In [33]:
if 'INJURY' in df.columns:
    df = df.drop(columns=['INJURY'])

df['INJURY'] = np.where(df['MAX_SEV_IR'].isin([1,2]), "Yes", "No")

b)

In [34]:
df12 = df.head(12)

i) Pivot table 

In [35]:
pivot = pd.crosstab(
    [df12['WEATHER_R'], df12['TRAF_CON_R']],
    df12['INJURY']
)

pivot

INJURY                No  Yes
WEATHER_R TRAF_CON_R         
1         0            1    2
          1            1    0
          2            1    0
2         0            5    1
          1            1    0

The pivot table shows how many accidents resulted in injuries and no injuries for each combination of WEATHER_R and TRAF_CON_R in the first 12 observations. For example, when WEATHER_R = 1 and TRAF_CON_R = 0, there were 2 injuries and 1 non-injury accident. When WEATHER_R = 2 and TRAF_CON_R = 0, there were 5 non-injuries and 1 injury. These counts allow us to see how accident outcomes vary across different weather and traffic conditions.

ii) Exacy Payes Conditional Probabilities

In [36]:
pivot['Total'] = pivot.sum(axis=1)

pivot['P(Injury=Yes)'] = pivot['Yes'] / pivot['Total']

pivot

INJURY                No  Yes  Total  P(Injury=Yes)
WEATHER_R TRAF_CON_R                               
1         0            1    2      3       0.666667
          1            1    0      1       0.000000
          2            1    0      1       0.000000
2         0            5    1      6       0.166667
          1            1    0      1       0.000000

Using the counts from the pivot table, I calculated the probability of INJURY = Yes for each combination of weather and traffic conditions. For example, when WEATHER_R = 1 and TRAF_CON_R = 0, the probability of injury is 0.667, meaning about 67% of accidents resulted in injury under those conditions in this sample. For several other combinations, the probability is 0, indicating that no injuries were observed for those conditions in the first 12 accidents.

iii) Classifications with Cutoffs = 0.5 

In [37]:
pivot['Prediction'] = np.where(pivot['P(Injury=Yes)'] >= 0.5, "Yes", "No")

pivot

INJURY                No  Yes  Total  P(Injury=Yes) Prediction
WEATHER_R TRAF_CON_R                                          
1         0            1    2      3       0.666667        Yes
          1            1    0      1       0.000000         No
          2            1    0      1       0.000000         No
2         0            5    1      6       0.166667         No
          1            1    0      1       0.000000         No

Using a cutoff value of 0.5, each combination of predictors was classified as either INJURY = Yes or INJURY = No. Only the combination WEATHER_R = 1 and TRAF_CON_R = 0 had a probability above 0.5, so it was classified as INJURY = Yes. All other combinations had probabilities below 0.5 and were therefore classified as INJURY = No.

iv) Manual Naive Payes Probability 

In [38]:
p_yes = (df12['INJURY']=="Yes").mean()

p_weather_given_yes = ((df12['WEATHER_R']==1) & (df12['INJURY']=="Yes")).sum() / (df12['INJURY']=="Yes").sum()

p_traf_given_yes = ((df12['TRAF_CON_R']==1) & (df12['INJURY']=="Yes")).sum() / (df12['INJURY']=="Yes").sum()

prob = p_yes * p_weather_given_yes * p_traf_given_yes

prob

np.float64(0.0)

The manually calculated Naive Bayes probability of INJURY = Yes given WEATHER_R = 1 and TRAF_CON_R = 1 was 0. This occurs because, in the first 12 observations, there were no accidents where both WEATHER_R = 1, TRAF_CON_R = 1, and INJURY = Yes occurred together. Since Naive Bayes multiplies conditional probabilities, the presence of a zero probability results in the final probability also being zero.

v) 

In [39]:
X = df12[['WEATHER_R','TRAF_CON_R']]
y = df12['INJURY']

X = X.astype(int)
y = y.map({'No':0,'Yes':1})

model = CategoricalNB()

model.fit(X,y)

probs = model.predict_proba(X)

pred = model.predict(X)

results = pd.DataFrame({
    "Actual": y,
    "Predicted": pred,
    "Prob_No": probs[:,0],
    "Prob_Yes": probs[:,1]
})

results

,Actual,Predicted,Prob_No,Prob_Yes
0,1,0,0.636364,0.363636
1,0,0,0.821229,0.178771
2,0,0,0.887324,0.112676
3,0,0,0.750000,0.250000
4,0,0,0.636364,0.363636
5,1,0,0.821229,0.178771
6,0,0,0.821229,0.178771
7,1,0,0.636364,0.363636
8,0,0,0.821229,0.178771
9,0,0,0.821229,0.178771


The Naive Bayes classifier predicted No Injury for all 12 observations, since the probability of injury for each observation was below 0.5. Although some accidents actually resulted in injuries, the model predicted the majority class because the estimated injury probabilities were relatively low. This shows that with such a small sample size 12 observations, the model does not have enough information to accurately distinguish injury cases.

C.

In [40]:
X = df[['WEATHER_R','TRAF_CON_R','SPD_LIM']]  # add other allowed predictors
y = df['INJURY']

y = y.map({'No':0,'Yes':1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.4, random_state=1
)

In [41]:
model = CategoricalNB()

model.fit(X_train, y_train)

pred = model.predict(X_valid)

c-ii

In [42]:
pd.DataFrame(conf_matrix,
             index=["Actual No","Actual Yes"],
             columns=["Pred No","Pred Yes"])

,Pred No,Pred Yes
Actual No,3228,5101
Actual Yes,2795,5750


The confusion matrix shows how well the Naive Bayes model predicted injury outcomes. The model correctly predicted 3228 accidents with no injury and 5750 accidents with injuries. However, it also made some mistakes: 5101 accidents were predicted as injuries when they were actually no injury, and 2795 accidents were predicted as no injury when they actually involved injuries. This indicates that the model makes a considerable number of classification errors.

c-iii

In [43]:
accuracy = accuracy_score(y_valid, pred)

error = 1 - accuracy

error


0.46793884082019677

The model’s error rate is approximately 0.468, meaning that about 46.8% of the validation observations were classified incorrectly. This shows that the model is correct a little more than half of the time, which suggests that the predictors WEATHER_R, TRAF_CON_R, and SPD_LIM have only moderate predictive power for determining whether an accident results in injury.

c-iv

In [45]:
naive_pred = [y_train.mode()[0]] * len(y_valid)

naive_error = 1 - accuracy_score(y_valid, naive_pred)

improvement = (naive_error - error) / naive_error * 100

improvement

5.198703325729379

The Naive Bayes model improves prediction accuracy by about 5.2% compared to a naive classifier that always predicts the most common outcome. This means that although the Naive Bayes model is not highly accurate, it still performs slightly better than simply guessing the majority class for every observation.

c-v

In [46]:
import pandas as pd

df = pd.read_csv("accidentsFull.csv")

df['INJURY'] = df['MAX_SEV_IR'].apply(lambda x: "Yes" if x in [1,2] else "No")

pivot_spd = pd.pivot_table(
    df,
    values='MAX_SEV_IR',       
    index='SPD_LIM',
    columns='INJURY',
    aggfunc='count',
    fill_value=0
)

pivot_spd['Total'] = pivot_spd.sum(axis=1)

pivot_spd['P_No'] = pivot_spd['No'] / pivot_spd['Total']
pivot_spd['P_Yes'] = pivot_spd['Yes'] / pivot_spd['Total']

print("Pivot table with counts and probabilities:")
print(pivot_spd)

print("\nConditional probabilities for SPD_LIM = 5:")
print(pivot_spd.loc[5, ['P_No','P_Yes']])

Pivot table with counts and probabilities:
INJURY     No   Yes  Total      P_No     P_Yes
SPD_LIM                                       
5           2     4      6  0.333333  0.666667
10         11    11     22  0.500000  0.500000
15         93    90    183  0.508197  0.491803
20        159    92    251  0.633466  0.366534
25       2245  1960   4205  0.533888  0.466112
30       1807  1908   3715  0.486406  0.513594
35       3994  4547   8541  0.467627  0.532373
40       1978  2326   4304  0.459572  0.540428
45       3240  3347   6587  0.491878  0.508122
50        844   821   1665  0.506907  0.493093
55       3306  3288   6594  0.501365  0.498635
60        727   931   1658  0.438480  0.561520
65       1371  1344   2715  0.504972  0.495028
70        818   636   1454  0.562586  0.437414
75        126   157    283  0.445230  0.554770

Conditional probabilities for SPD_LIM = 5:
INJURY
P_No     0.333333
P_Yes    0.666667
Name: 5, dtype: float64


For SPD_LIM = 5, 2 out of 6 accidents did not result in injury, and 4 out of 6 did, giving conditional probabilities:
P(INJURY = No | SPD_LIM = 5)=0.33,P(INJURY = Yes | SPD_LIM = 5)=0.67
A probability of zero can occur in Naive Bayes if no accidents with a certain speed limit have a particular outcome. This is called the zero-frequency problem and is usually handled with Laplace smoothing to avoid zero probabilities.

2.

In [48]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, accuracy_score

In [49]:
df = pd.read_csv("ToyotaCorolla.csv")
df.head()

,Id,Model,Price,Age_08_04,Mfg_Month,Mfg_Year,KM,Fuel_Type,HP,Met_Color,...,Powered_Windows,Power_Steering,Radio,Mistlamps,Sport_Model,Backseat_Divider,Metallic_Rim,Radio_cassette,Parking_Assistant,Tow_Bar
0,1,TOYOTA Corolla 2.0 D4D HATCHB TERRA 2/3-Doors,13500,23,10,2002,46986,Diesel,90,1,...,1,1,0,0,0,1,0,0,0,0
1,2,TOYOTA Corolla 2.0 D4D HATCHB TERRA 2/3-Doors,13750,23,10,2002,72937,Diesel,90,1,...,0,1,0,0,0,1,0,0,0,0
2,3,TOYOTA Corolla 2.0 D4D HATCHB TERRA 2/3-Doors,13950,24,9,2002,41711,Diesel,90,1,...,0,1,0,0,0,1,0,0,0,0
3,4,TOYOTA Corolla 2.0 D4D HATCHB TERRA 2/3-Doors,14950,26,7,2002,48000,Diesel,90,0,...,0,1,0,0,0,1,0,0,0,0
4,5,TOYOTA Corolla 2.0 D4D HATCHB SOL 2/3-Doors,13750,30,3,2002,38500,Diesel,90,0,...,1,1,0,1,0,1,0,0,0,0


In [63]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("ToyotaCorolla.csv")

df = pd.get_dummies(df, columns=['Fuel_Type'], drop_first=True)

predictors = ['Age_08_04', 'KM', 'HP', 'Automatic', 'Doors', 
              'Quarterly_Tax', 'Mfr_Guarantee', 'Guarantee_Period', 'Airco', 
              'Automatic_airco', 'CD_Player', 'Powered_Windows', 'Sport_Model', 
              'Tow_Bar',
              'Fuel_Type_Diesel', 'Fuel_Type_Petrol']  

X = df[predictors]
y = df['Price']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.4, random_state=1)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_valid.shape, y_valid.shape)

Training set: (861, 16) (861,)
Validation set: (575, 16) (575,)


a-i

In [64]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import pandas as pd

rt_full = DecisionTreeRegressor(random_state=1)
rt_full.fit(X_train, y_train)

importances = pd.Series(rt_full.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Feature Importances:\n", importances.head(4))  


Feature Importances:
 Age_08_04          0.844867
HP                 0.053789
KM                 0.049601
Automatic_airco    0.013358
dtype: float64


The most important predictors for the car’s price are Age_08_04 (by far the most important), followed by HP, KM, and Automatic_airco. This means the car’s age has the largest impact on price, while horsepower and kilometers driven also influence it to a smaller extent.

a-ii

In [65]:
y_train_pred = rt_full.predict(X_train)
y_valid_pred = rt_full.predict(X_valid)

mse_train = mean_squared_error(y_train, y_train_pred)
mse_valid = mean_squared_error(y_valid, y_valid_pred)
print(f"Training MSE: {mse_train:.2f}")
print(f"Validation MSE: {mse_valid:.2f}")

Training MSE: 0.00
Validation MSE: 2227068.13


The training MSE is 0, meaning the full-grown tree fits the training data perfectly. However, the validation MSE is very large (2,227,068), showing that the model overfits the training data and does not generalize well to new data.


a-iii

In [66]:
from sklearn.model_selection import GridSearchCV

param_grid = {'max_depth': list(range(2, 11)), 'min_samples_split': [2, 5, 10]}

grid = GridSearchCV(DecisionTreeRegressor(random_state=1),
                    param_grid, cv=5,
                    scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)

rt_tuned = grid.best_estimator_
print("Best Parameters:", grid.best_params_)

y_valid_pred_tuned = rt_tuned.predict(X_valid)
mse_valid_tuned = mean_squared_error(y_valid, y_valid_pred_tuned)
print(f"Tuned Tree Validation MSE: {mse_valid_tuned:.2f}")

Best Parameters: {'max_depth': 5, 'min_samples_split': 5}
Tuned Tree Validation MSE: 1383149.16


After tuning, the regression tree is limited to a maximum depth of 5 and a minimum of 5 samples per split. This reduces overfitting: the validation MSE decreased from 2,227,068 to 1,383,149. Training MSE increases slightly, but the model now generalizes better to new data.

b-i

In [67]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df['Binned_Price'] = pd.cut(df['Price'], bins=20, labels=False)

y_class = df['Binned_Price']
X_train_c, X_valid_c, y_train_c, y_valid_c = train_test_split(X, y_class, test_size=0.4, random_state=1)

ct_full = DecisionTreeClassifier(random_state=1)
ct_full.fit(X_train_c, y_train_c)

param_grid_c = {'max_depth': list(range(2, 11)), 'min_samples_split': [2, 5, 10]}
grid_c = GridSearchCV(DecisionTreeClassifier(random_state=1),
                      param_grid_c, cv=5, scoring='accuracy')
grid_c.fit(X_train_c, y_train_c)

ct_tuned = grid_c.best_estimator_
print("Tuned Classification Tree Params:", grid_c.best_params_)

y_valid_pred_c = ct_tuned.predict(X_valid_c)
acc_valid_c = accuracy_score(y_valid_c, y_valid_pred_c)
print(f"Tuned Classification Tree Accuracy: {acc_valid_c:.2f}")

C:\Users\jamal\OneDrive\Documents\IDS\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Tuned Classification Tree Params: {'max_depth': 5, 'min_samples_split': 2}
Tuned Classification Tree Accuracy: 0.48


The tuned classification tree has a maximum depth of 5 and a minimum of 2 samples per split. Its validation accuracy is 0.48, meaning it correctly predicts the price bin for about 48% of cars. The CT is simpler than the full tree and focuses on predicting price ranges rather than exact prices.

b-ii

In [68]:
new_car = pd.DataFrame({
    'Age_08_04': [5],
    'KM': [50000],
    'HP': [90],
    'Automatic': [0],
    'Doors': [5],
    'Quarterly_Tax': [200],
    'Mfr_Guarantee': [1],
    'Guarantee_Period': [12],
    'Airco': [1],
    'Automatic_airco': [0],
    'CD_Player': [1],
    'Powered_Windows': [1],
    'Sport_Model': [0],
    'Tow_Bar': [0],
    'Fuel_Type_Diesel': [0],
    'Fuel_Type_Petrol': [1]
})

pred_price_rt = rt_tuned.predict(new_car)
print(f"Predicted Price (RT): {pred_price_rt[0]:.2f}")

pred_bin_ct = ct_tuned.predict(new_car)
print(f"Predicted Price Bin (CT): {pred_bin_ct[0]}")

bins = pd.cut(df['Price'], bins=20)
bin_edges = bins.cat.categories
pred_bin_index = pred_bin_ct[0]
approx_price_ct = (bin_edges[pred_bin_index].left + bin_edges[pred_bin_index].right) / 2
print(f"Approx. Price from CT Bin: {approx_price_ct:.2f}")

Predicted Price (RT): 16880.77
Predicted Price Bin (CT): 11
Approx. Price from CT Bin: 20536.25


The tuned regression tree predicts an exact price of $16,880.77 for the new car. The tuned classification tree predicts it falls in price bin 11, which corresponds to an approximate price of $20,536.25. The RT gives a precise value, while the CT provides a price range.